In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os
from dtw import dtw
import matplotlib.pyplot as plt
import io
import requests
import json
import time


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [2]:
well_sites = r"C:\Users\romin\OneDrive\Groundwater\RpSy Data\Site information for all selected wells.xlsx"
state_boundaries = r"C:\Users\romin\OneDrive\Groundwater\cb_2022_us_state_500k"
recharge_dir = r"C:/Users/romin/OneDrive/Groundwater/RpSy Data"
daymet_dir = "./daymet"

NE_states = [
    "Connecticut", "Maine", "Massachusetts", "New Hampshire",
    "Rhode Island", "Vermont", "New Jersey", "New York", "Pennsylvania",
]

# ============================================================
# Load wells and filter to NE states
# ============================================================

def load_sites(path=well_sites):
    df = pd.read_excel(path)
    df = df.rename(columns={
        "ID": "usgs_id",
        "Lat": "lat",
        "Long": "lon",
        "depth (m)": "depth",
    })
    return df

def select_ne_wells(df_sites, states_shp=state_boundaries):
    states = gpd.read_file(states_shp)
    ne = states[states["NAME"].isin(NE_states)].set_crs(epsg=4326, allow_override=True)
    gdf_sites = gpd.GeoDataFrame(
        df_sites,
        geometry=gpd.points_from_xy(df_sites["lon"], df_sites["lat"]),
        crs="EPSG:4326",
    )
    joined = gpd.sjoin(gdf_sites, ne[["NAME", "geometry"]], how="inner", predicate="within")
    joined = joined.rename(columns={"NAME": "state"}).drop(columns=["index_right"])
    return joined

df_sites = load_sites()
df_ne_wells = select_ne_wells(df_sites)

In [3]:
def get_start_end_dates(df, date_col="Date"):
    df[date_col] = pd.to_datetime(df[date_col])
    return df[date_col].min(), df[date_col].max() 

start_dates = []
end_dates = []
for wid in df_ne_wells["usgs_id"]:
    df = pd.read_csv(f"{recharge_dir}/{wid}.csv")
    start_date, end_date = get_start_end_dates(df)
    start_dates.append(start_date)
    end_dates.append(end_date)

df_ne_wells["start_date"] = start_dates
df_ne_wells["end_date"] = end_dates
df_ne_wells["record_length"] = (df_ne_wells["end_date"] - df_ne_wells["start_date"]).dt.days / 365.25

In [4]:
eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Wells with >=5 year records: {len(eligible_wells)}")

selected_wells = eligible_wells.sample(n=10, random_state=42)
print(selected_wells[["usgs_id", "record_length", "start_date", "end_date"]])

Wells with >=5 year records: 163
             usgs_id  record_length start_date   end_date
426  425803077151201      18.995209 2003-10-02 2022-09-30
403  421746074180201      15.994524 2006-10-02 2022-09-30
422  424520070562401      13.993155 1985-10-02 1999-09-30
306  404639074230001      12.993840 2009-10-02 2022-09-30
356  414330076280501      23.994524 1998-10-02 2022-09-30
274  400229075104601       9.993155 2012-10-02 2022-09-30
458  444904074455201      19.994524 2002-10-02 2022-09-30
302  404140077354001      16.996578 1999-10-02 2016-09-30
364  415228070554601       7.994524 2000-10-02 2008-09-30
439  434217073010601       5.993155 2016-10-02 2022-09-30


In [5]:
# DTW helper functions
def get_events(values, threshold):
    """Return start/end index pairs for runs of consecutive values > threshold."""
    above = (values > threshold).astype(int)
    padded = np.concatenate(([0], above, [0]))
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1
    return starts, ends

def precip_recharge_event_table(df, precip_col, recharge_col, alignment, precip_threshold=0.0):
    """Return a table of precipitation and recharge events based on DTW alignment."""
    dates = df.index
    precip_vals = df[precip_col].values
    recharge_vals = df[recharge_col].values
    starts, ends = get_events(precip_vals, precip_threshold)

    rows = []
    prev_recharge_idx = None
    for s, e in zip(starts, ends):
        precip_idx = np.arange(s, e + 1)
        mask = np.isin(alignment.index2, precip_idx)
        recharge_idx = np.unique(alignment.index1[mask])

        if prev_recharge_idx is not None and recharge_idx.size > 0:
            test = recharge_idx > prev_recharge_idx.max()
            recharge_idx = recharge_idx[test]   # dropped silently, no print

        row = {
            "precip_start_date": dates[s],
            "precip_end_date": dates[e],
            "precip_total": precip_vals[precip_idx].sum(),
            "precip_peak": precip_vals[precip_idx].max(),
        }
        if recharge_idx.size == 0:
            row.update({
                "recharge_start_date": pd.NaT,
                "recharge_end_date": pd.NaT,
                "recharge_total": np.nan,
                "recharge_peak": np.nan,
                "n_recharge_days": 0,
            })
        else:
            row.update({
                "recharge_start_date": dates[recharge_idx.min()],
                "recharge_end_date": dates[recharge_idx.max()],
                "recharge_total": recharge_vals[recharge_idx].sum(),
                "recharge_peak": recharge_vals[recharge_idx].max(),
                "n_recharge_days": recharge_idx.size,
            })
        rows.append(row)
        prev_recharge_idx = recharge_idx if recharge_idx.size > 0 else prev_recharge_idx

    return pd.DataFrame(rows)

def check_overlapping_recharge_events(event_table):
    """Check for overlapping recharge events in the event table."""
    sorted_table = event_table.sort_values("recharge_start_date")
    overlaps = []
    for i in range(len(sorted_table) - 1):
        current_end = sorted_table.iloc[i]["recharge_end_date"]
        next_start = sorted_table.iloc[i + 1]["recharge_start_date"]
        if pd.notna(current_end) and pd.notna(next_start) and current_end >= next_start:
            overlap_days = (current_end - next_start).days + 1
            overlaps.append((i, current_end, next_start, overlap_days))
    return overlaps

In [6]:
os.makedirs("dtw_event_tables", exist_ok=True)

eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Total wells with >=5 year records: {len(eligible_wells)}")

def run_dtw_pipeline(site_id, recharge_dir=recharge_dir, daymet_dir=daymet_dir,
                      out_dir="dtw_event_tables",
                      precip_threshold=2.5, min_lag=0, max_lag=5):
    recharge_path = f"{recharge_dir}/{site_id}.csv"
    precip_path = f"{daymet_dir}/{site_id}.csv"

    if not os.path.exists(recharge_path) or not os.path.exists(precip_path):
        return None, "missing file"

    df_recharge = pd.read_csv(recharge_path)
    df_precip = pd.read_csv(precip_path)

    df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)

    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")

    if len(df) < 30:
        return None, "too short"

    recharge = df["RpSy (m)"].values
    precip = df["prcp (mm/day)"].values

    if np.std(recharge) == 0 or np.std(precip) == 0:
        return None, "zero variance"

    recharge_norm = recharge / np.std(recharge)
    precip_norm = precip / np.std(precip)

    def causal_window(iw, jw, query_size, reference_size, min_lag=min_lag, max_lag=max_lag, **kwargs):
        lag = iw - jw
        ok = lag >= min_lag
        if max_lag is not None:
            ok = ok & (lag <= max_lag)
        return ok

    try:
        alignment = dtw(
            recharge_norm, precip_norm,
            step_pattern="symmetric2",
            window_type=causal_window,
            window_args={"min_lag": min_lag, "max_lag": max_lag},
            keep_internals=True,
            open_begin=False,
            open_end=False,
        )
    except Exception as e:
        return None, f"dtw failed: {e}"

    event_table = precip_recharge_event_table(
        df, "prcp (mm/day)", "RpSy (m)", alignment, precip_threshold=precip_threshold
    )

    out_path = f"{out_dir}/{site_id}_dtw_event_table.csv"
    event_table.to_csv(out_path, index=False)

    return event_table, "ok"


all_event_tables = {}
failed_wells = {}

for i, site_id in enumerate(eligible_wells["usgs_id"]):
    site_id = str(site_id)
    print(f"\rProcessing well {i+1}/{len(eligible_wells)}...", end="", flush=True)

    out_path = f"dtw_event_tables/{site_id}_dtw_event_table.csv"
    if os.path.exists(out_path):
        all_event_tables[site_id] = pd.read_csv(out_path)
        continue

    table, status = run_dtw_pipeline(site_id)
    if table is not None:
        all_event_tables[site_id] = table
    else:
        failed_wells[site_id] = status

print(f"\n\nCompleted: {len(all_event_tables)} of {len(eligible_wells)} wells")
print(f"Failed/skipped: {len(failed_wells)}")
if failed_wells:
    from collections import Counter
    reasons = Counter(failed_wells.values())
    print("Failure reasons:", dict(reasons))

Total wells with >=5 year records: 163
Processing well 163/163...

Completed: 163 of 163 wells
Failed/skipped: 0


In [7]:
import matplotlib
matplotlib.use('QtAgg')
import matplotlib.pyplot as plt

def plot_dtw_events(site_id, event_table, recharge_dir=recharge_dir, daymet_dir=daymet_dir):
    df_recharge = pd.read_csv(f"{recharge_dir}/{site_id}.csv")
    df_precip = pd.read_csv(f"{daymet_dir}/{site_id}.csv")
    df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)
    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")

    date_cols = ["recharge_start_date", "recharge_end_date", "precip_start_date", "precip_end_date"]
    event_table[date_cols] = event_table[date_cols].apply(pd.to_datetime)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    color = 'tab:red'
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Recharge (m)', color=color)
    recharge_max = df["RpSy (m)"].max()
    ax1.set_ylim(0, recharge_max * 1.5)
    ax1.plot(df.index, df["RpSy (m)"], color=color)
    ax1.tick_params(axis='y', labelcolor=color)
   
    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Precipitation (mm)', color=color)
    ax2.plot(df.index, df["prcp (mm/day)"], color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    precip_max = df["prcp (mm/day)"].max()
    ax2.set_ylim(precip_max * 1.5, 0)
    axmax = ax1.get_ylim()[1]
    for _, row in event_table.iterrows():
        if pd.notna(row["recharge_start_date"]) and pd.notna(row["recharge_end_date"]):
            x = [row["recharge_end_date"], row["recharge_start_date"],
                 row["precip_start_date"], row["precip_end_date"]]
            y = [0, 0, axmax, axmax]
            ax1.fill(x, y, color='gray', alpha=0.2)
    ax1.set_title(f"Well {site_id}")
    plt.show()


In [8]:
all_ratios = []

for well_id, table in all_event_tables.items():
    df = table.dropna(subset=["precip_total", "recharge_total"]).copy()
    df = df[df["precip_total"] >= 10]  # magnitude floor, testing the ratio cutoff separately
    df["implied_ratio"] = (df["recharge_total"] * 1000) / df["precip_total"]
    df["well_id"] = well_id
    all_ratios.append(df[["well_id", "precip_total", "recharge_total", "implied_ratio"]])

all_ratios_df = pd.concat(all_ratios, ignore_index=True)

print(all_ratios_df["implied_ratio"].describe())
print("\nPercentiles:")
for p in [50, 75, 90, 90, 97.5, 99, 99.5, 99.9]:
    print(f"  {p}th: {all_ratios_df['implied_ratio'].quantile(p/100):.2f}")

count    102828.000000
mean          6.598049
std          13.428459
min           0.000000
25%           0.896236
50%           2.832505
75%           7.029045
max         593.564680
Name: implied_ratio, dtype: float64

Percentiles:
  50th: 2.83
  75th: 7.03
  90th: 14.96
  90th: 14.96
  97.5th: 37.79
  99th: 61.63
  99.5th: 84.88
  99.9th: 154.00


#### Ratio distribution

In [9]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_ratios_df["implied_ratio"], bins=200, color="tab:blue", alpha=0.7)
ax.set_xlim(0, 50)
ax.set_xlabel("Implied recharge/precip ratio")
ax.set_ylabel("Number of events")
ax.set_title("Distribution of Implied Ratios, All Wells, All Events")
ax.grid(alpha=0.3)
plt.show()

In [11]:
max_ratio = 12
kept = all_ratios_df[all_ratios_df["implied_ratio"] <= max_ratio]
dropped = all_ratios_df[all_ratios_df["implied_ratio"] > max_ratio]

print(f"Cutoff: ratio <= {max_ratio}")
print(f"Kept: {len(kept)} of {len(all_ratios_df)} events ({len(kept)/len(all_ratios_df)*100:.1f}%)")
print(f"Dropped: {len(dropped)} events ({len(dropped)/len(all_ratios_df)*100:.1f}%)")

pct_at_12 = (all_ratios_df["implied_ratio"] <= 12).mean() * 100
print(f"\nA ratio of 12 corresponds to roughly the {pct_at_12:.1f}th percentile of all events")

Cutoff: ratio <= 12
Kept: 89042 of 102828 events (86.6%)
Dropped: 13786 events (13.4%)

A ratio of 12 corresponds to roughly the 86.6th percentile of all events


In [12]:
MAX_RATIO = 12

def filter_events_by_ratio(event_table, max_ratio=MAX_RATIO,
                              precip_col="precip_total", recharge_col="recharge_total"):
    """
    Drops events with an implausible recharge/precip ratio.
    No minimum magnitude filter -- every event with valid precip
    and recharge values is included unless its ratio is too extreme.
    """
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
    df = df[df[precip_col] > 0]  # avoid divide-by-zero
    df["implied_ratio"] = (df[recharge_col] * 1000) / df[precip_col]
    df = df[df["implied_ratio"] <= max_ratio]
    return df

In [13]:
def plot_cumulative_precip_recharge(event_table, well_id, max_ratio=MAX_RATIO,
                                       precip_col="precip_total", recharge_col="recharge_total"):
    """
    Sorts events SMALLEST to LARGEST by precipitation, then plots
    the cumulative fraction of total precipitation (x) against the
    cumulative fraction of total recharge (y).
    """
    df = filter_events_by_ratio(event_table, max_ratio=max_ratio,
                                   precip_col=precip_col, recharge_col=recharge_col)

    if len(df) == 0:
        print(f"No events found for {well_id}. Skipping.")
        return

    df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)

    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    df["cum_precip_frac"] = df[precip_col].cumsum() / total_precip
    df["cum_recharge_frac"] = df[recharge_col].cumsum() / total_recharge

    x = np.concatenate(([0], df["cum_precip_frac"].values))
    y = np.concatenate(([0], df["cum_recharge_frac"].values))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x, y, marker="o", color="black", markersize=3, label="Cumulative curve")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6, label="1:1 line")
    ax.set_xlabel("Cumulative fraction of total precipitation")
    ax.set_ylabel("Cumulative fraction of total recharge")
    ax.set_title(f"Well {well_id} (n={len(df)}, max ratio = {max_ratio})")
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


site_id = list(all_event_tables.keys())[12]
plot_cumulative_precip_recharge(all_event_tables[site_id], site_id)

In [14]:
def lorenz_curve(values):
    """
    x = cumulative fraction of events (smallest to largest),
    y = cumulative fraction of the total. Returns x, y, Gini.
    """
    sorted_vals = np.sort(values)  # ascending: smallest first
    n = len(sorted_vals)
    cum_vals = np.cumsum(sorted_vals)
    cum_frac = cum_vals / cum_vals[-1]
    x = np.concatenate(([0], np.arange(1, n + 1) / n))
    y = np.concatenate(([0], cum_frac))
    trapezoid_func = getattr(np, "trapezoid", None) or np.trapz
    gini = 1 - 2 * trapezoid_func(y, x)
    return x, y, gini


def plot_lorenz(event_table, well_id, value_col, label, max_ratio=MAX_RATIO,
                  precip_col="precip_total", recharge_col="recharge_total"):
    df = filter_events_by_ratio(event_table, max_ratio=max_ratio,
                                   precip_col=precip_col, recharge_col=recharge_col)

    if len(df) < 2:
        print(f"Too few events for {well_id}")
        return None

    x, y, gini = lorenz_curve(df[value_col].values)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x, y, color="black", linewidth=2)
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6, label="Perfect equality")
    ax.fill_between(x, y, x, alpha=0.15, color="tab:red")
    ax.set_xlabel(f"Cumulative fraction of {label} events (smallest to largest)")
    ax.set_ylabel(f"Cumulative fraction of total {label}")
    ax.set_title(f"Well {well_id} — {label} Lorenz Curve\nGini = {gini:.3f} (n={len(df)})")
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
    return gini


# Test on one well
site_id = list(all_event_tables.keys())[0]
plot_lorenz(all_event_tables[site_id], site_id, "precip_total", "Precipitation")
plot_lorenz(all_event_tables[site_id], site_id, "recharge_total", "Recharge")

np.float64(0.6798610686274718)

In [15]:
from scipy.optimize import curve_fit

def kumaraswamy_cdf(x, a, b):
    """Kumaraswamy CDF: F(x) = 1 - (1 - x^a)^b"""
    return 1 - (1 - np.clip(x, 0, 1) ** a) ** b


def fit_kumaraswamy(x_data, y_data):
    """Fits a and b to match the observed cumulative curve. Returns (a, b) or None."""
    try:
        popt, _ = curve_fit(kumaraswamy_cdf, x_data, y_data, p0=[1, 1], maxfev=5000)
        return popt
    except Exception:
        return None


def plot_kumaraswamy_fit(event_table, well_id, max_ratio=MAX_RATIO,
                            precip_col="precip_total", recharge_col="recharge_total"):
    """
    Fits a Kumaraswamy curve to this well's cumulative precip-vs-recharge
    curve (smallest to largest, matching plot_cumulative_precip_recharge),
    and overlays the fitted curve for comparison.
    """
    df = filter_events_by_ratio(event_table, max_ratio=max_ratio,
                                   precip_col=precip_col, recharge_col=recharge_col)

    if len(df) < 5:
        print(f"Not enough events for well {well_id}. Skipping.")
        return None

    df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)

    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    x_data = np.concatenate(([0], (df[precip_col].cumsum() / total_precip).values))
    y_data = np.concatenate(([0], (df[recharge_col].cumsum() / total_recharge).values))

    params = fit_kumaraswamy(x_data, y_data)
    if params is None:
        print(f"Fit failed for well {well_id}")
        return None

    a, b = params
    x_smooth = np.linspace(0, 1, 100)
    y_smooth = kumaraswamy_cdf(x_smooth, a, b)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(x_data, y_data, "o", color="black", markersize=3, alpha=0.5, label="Actual cumulative curve")
    ax.plot(x_smooth, y_smooth, color="red", linewidth=2, label=f"Kumaraswamy fit (a={a:.2f}, b={b:.2f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.5, label="1:1 line")

    ax.set_xlabel("Cumulative fraction of precipitation\n(smallest events first)")
    ax.set_ylabel("Cumulative fraction of recharge")
    ax.set_title(f"Well {well_id} — Kumaraswamy Fit (n={len(df)}, max ratio = {max_ratio})")
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

    return a, b


site_id = list(all_event_tables.keys())[0]
a, b = plot_kumaraswamy_fit(all_event_tables[site_id], site_id)

In [16]:
def fit_kumaraswamy_curve(event_table, max_ratio=MAX_RATIO,
                             precip_col="precip_total", recharge_col="recharge_total"):
    df = filter_events_by_ratio(event_table, max_ratio=max_ratio,
                                   precip_col=precip_col, recharge_col=recharge_col)
    if len(df) < 5:
        return None
    df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)
    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    x_data = np.concatenate(([0], (df[precip_col].cumsum() / total_precip).values))
    y_data = np.concatenate(([0], (df[recharge_col].cumsum() / total_recharge).values))
    return fit_kumaraswamy(x_data, y_data)


kuma_results = []
for well_id, table in all_event_tables.items():
    params = fit_kumaraswamy_curve(table)
    if params is not None:
        kuma_results.append({"well_id": well_id, "a": params[0], "b": params[1]})

kuma_df = pd.DataFrame(kuma_results)
print(f"Successfully fit: {len(kuma_df)} of {len(all_event_tables)} wells")
print(kuma_df.describe())

Successfully fit: 163 of 163 wells
                a           b
count  163.000000  163.000000
mean     1.049351    0.954009
std      0.182929    0.169623
min      0.643060    0.604371
25%      0.925603    0.825019
50%      1.045717    0.931134
75%      1.182335    1.046133
max      1.501700    1.616752


In [17]:
pa_wells = df_ne_wells[df_ne_wells["state"] == "Pennsylvania"]["usgs_id"].astype(str).tolist()
pa_wells_in_data = [w for w in pa_wells if w in all_event_tables]
print(f"PA wells with event tables: {len(pa_wells_in_data)}")


def plot_all_kumaraswamy_curves(well_ids, all_event_tables, title="Kumaraswamy Fits by Well"):
    """
    One figure, every well's fitted curve overlaid as its own colored line.
    """
    fig, ax = plt.subplots(figsize=(9, 8))
    x_smooth = np.linspace(0, 1, 200)

    try:
        cmap = plt.colormaps.get_cmap("tab20")
    except AttributeError:
        cmap = plt.cm.get_cmap("tab20")

    n = max(len(well_ids), 1)

    for i, well_id in enumerate(well_ids):
        params = fit_kumaraswamy_curve(all_event_tables[well_id])
        if params is None:
            continue

        a, b = params
        y_smooth = kumaraswamy_cdf(x_smooth, a, b)
        color = cmap(i / n)
        ax.plot(x_smooth, y_smooth, color=color, linewidth=1.5,
                label=f"{well_id} (a={a:.2f}, b={b:.2f})")

    ax.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.4, label="1:1 line")

    ax.set_xlabel("Cumulative fraction of precipitation (smallest events first)")
    ax.set_ylabel("Cumulative fraction of recharge")
    ax.set_title(title)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=6, loc="upper left", bbox_to_anchor=(1.02, 1))
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_all_kumaraswamy_curves(pa_wells_in_data, all_event_tables, title="Kumaraswamy Fits — Pennsylvania Wells")

PA wells with event tables: 39


In [18]:
def classify_shape(a, b):
    if a > 1 and b > 1:
        return "S-shaped (mid-size storms dominate)"
    elif a > 1 and b <= 1:
        return "Late-rise (few biggest storms dominate)"
    elif a <= 1 and b > 1:
        return "Early-rise (small storms dominate)"
    else:
        return "U-shaped (both extremes matter, middle doesn't)"

kuma_df["shape"] = kuma_df.apply(lambda row: classify_shape(row["a"], row["b"]), axis=1)
print(kuma_df["shape"].value_counts())

shape
Late-rise (few biggest storms dominate)            68
U-shaped (both extremes matter, middle doesn't)    43
S-shaped (mid-size storms dominate)                34
Early-rise (small storms dominate)                 18
Name: count, dtype: int64


#### Well's with the lowest MTPI will have the highest ratio of recharge to precipitation.

In [19]:
# Per-well summary: what's the max implied ratio at each well?
well_max_ratio = all_ratios_df.groupby("well_id")["implied_ratio"].max().reset_index()
well_max_ratio.columns = ["well_id", "max_ratio"]

# The 90th percentile cutoff, across all wells' max ratios
threshold_90 = well_max_ratio["max_ratio"].quantile(0.90)
print(f"90th percentile of max ratio across wells: {threshold_90:.2f}")

flagged_wells = well_max_ratio[well_max_ratio["max_ratio"] >= threshold_90]
print(f"Wells flagged (90th percentile or higher): {len(flagged_wells)}")
print(flagged_wells.sort_values("max_ratio", ascending=False))

90th percentile of max ratio across wells: 113.10
Wells flagged (90th percentile or higher): 17
             well_id   max_ratio
14   394430077225001  593.564680
51   404140077354001  546.712383
99   414640077493801  358.026329
54   404556077525101  332.798354
84   413026076352901  290.356930
41   402512074414301  269.795204
73   411833075133601  202.445255
56   404708076070701  181.983303
75   412020079133901  169.518855
153  443647070552303  161.569315
155  444302070252401  147.211693
65   410449074483301  128.050122
31   400916076492301  125.914954
95   414330076280501  120.088595
52   404239076362001  119.508368
39   402255076422001  118.474463
28   400217078281901  113.767807


In [20]:
master_csv = pd.read_csv("master_well_summary.csv")  # adjust path if it's saved elsewhere
print(master_csv.columns.tolist())

['well_id', 'mtpi', 'elevation_m', 'mean_depth_to_water_m', 'clay', 'sand', 'silt', 'state', 'precip_gini', 'recharge_gini', 'kuma_a', 'kuma_b', 'lat', 'lon', 'nlcd_code', 'landcover', 'landcover_simple', 'n_events_x', 'n_events_y', 'total_precip_mm', 'total_recharge_mm', 'recharge_efficiency']


In [21]:
flagged_ids = flagged_wells["well_id"].tolist()

flagged_mtpi = master_csv[master_csv["well_id"].astype(str).isin(flagged_ids)][["well_id", "mtpi"]]
other_mtpi = master_csv[~master_csv["well_id"].astype(str).isin(flagged_ids)][["well_id", "mtpi"]]

print(f"Flagged wells mTPI: mean={flagged_mtpi['mtpi'].mean():.2f}, median={flagged_mtpi['mtpi'].median():.2f}, n={len(flagged_mtpi)}")
print(f"Other wells mTPI: mean={other_mtpi['mtpi'].mean():.2f}, median={other_mtpi['mtpi'].median():.2f}, n={len(other_mtpi)}")

Flagged wells mTPI: mean=-8.06, median=-2.00, n=17
Other wells mTPI: mean=-5.36, median=-2.00, n=146


In [22]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot([flagged_mtpi["mtpi"].dropna(), other_mtpi["mtpi"].dropna()],
           label=["Flagged (90th %ile+ ratio)", "Other wells"])
ax.set_ylabel("mTPI (negative = valley, positive = ridge)")
ax.set_title("Topographic Position: Extreme-Ratio Wells vs. Others")
ax.grid(alpha=0.3, axis="y")
plt.show()

In [23]:
from scipy import stats

t_stat, p_val = stats.ttest_ind(flagged_mtpi["mtpi"].dropna(), other_mtpi["mtpi"].dropna())
print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")
print(f"{'Significant difference' if p_val < 0.05 else 'No significant difference'}")

t-test: t=-0.832, p=0.4065
No significant difference


In [24]:
print(flagged_mtpi.sort_values("mtpi"))

             well_id  mtpi
99   414640077493801   -60
84   413026076352901   -54
56   404708076070701   -46
95   414330076280501   -23
28   400217078281901    -7
31   400916076492301    -3
39   402255076422001    -3
153  443647070552303    -3
41   402512074414301    -2
73   411833075133601    -1
51   404140077354001     0
54   404556077525101     0
65   410449074483301     0
52   404239076362001     7
75   412020079133901    15
155  444302070252401    21
14   394430077225001    22


In [25]:
def get_nlcd_landcover(lat, lon, year=2019):
    """
    Pulls the NLCD land cover class for one point, via Earth Engine.
    """
    dataset = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").filter(
        ee.Filter.eq("system:index", str(year))
    ).first()
    landcover = dataset.select("landcover")

    point = ee.Geometry.Point([lon, lat])
    value = landcover.reduceRegion(ee.Reducer.first(), point, scale=30).get("landcover")
    return value.getInfo()

In [26]:
NLCD_CODES = {
    11: "Open Water", 12: "Perennial Ice/Snow",
    21: "Developed, Open Space", 22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity", 24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest", 42: "Evergreen Forest", 43: "Mixed Forest",
    51: "Dwarf Scrub", 52: "Shrub/Scrub",
    71: "Grassland/Herbaceous", 72: "Sedge/Herbaceous", 73: "Lichens", 74: "Moss",
    81: "Pasture/Hay", 82: "Cultivated Crops",
    90: "Woody Wetlands", 95: "Emergent Herbaceous Wetlands",
}

def simplify_landcover(code):
    """Groups the detailed NLCD codes into broader categories."""
    if code is None:
        return None
    if code in [11, 12]:
        return "Water"
    if code in [21, 22, 23, 24]:
        return "Developed"
    if code == 31:
        return "Barren"
    if code in [41, 42, 43]:
        return "Forest"
    if code in [51, 52]:
        return "Shrub"
    if code in [71, 72, 73, 74]:
        return "Grassland"
    if code in [81, 82]:
        return "Agriculture"
    if code in [90, 95]:
        return "Wetland"
    return "Other"

In [27]:
print(master_csv["landcover"].value_counts())

landcover
Developed, Open Space           31
Developed, Low Intensity        22
Developed, Medium Intensity     19
Deciduous Forest                19
Pasture/Hay                     18
Mixed Forest                    16
Woody Wetlands                  13
Developed, High Intensity       10
Evergreen Forest                 6
Cultivated Crops                 4
Grassland/Herbaceous             2
Barren Land                      1
Emergent Herbaceous Wetlands     1
Shrub/Scrub                      1
Name: count, dtype: int64


In [28]:
print(master_csv["landcover_simple"].value_counts())

landcover_simple
Developed      82
Forest         41
Agriculture    22
Wetland        14
Grassland       2
Barren          1
Shrub           1
Name: count, dtype: int64


In [29]:
def plot_kuma_by_landcover(master_csv):
    """
    Scatter: Kumaraswamy 'a' on x, 'b' on y, one dot per well,
    colored by dominant land cover category.
    """
    df = master_csv.dropna(subset=["kuma_a", "kuma_b", "landcover_simple"])

    categories = df["landcover_simple"].unique()
    cmap = plt.colormaps.get_cmap("tab10")

    fig, ax = plt.subplots(figsize=(10, 8))

    for i, category in enumerate(categories):
        subset = df[df["landcover_simple"] == category]
        ax.scatter(subset["kuma_a"], subset["kuma_b"], color=cmap(i / max(len(categories), 1)),
                   s=60, edgecolor="black", linewidth=0.4, alpha=0.7,
                   label=f"{category} (n={len(subset)})")

    ax.set_xlabel("Kumaraswamy a")
    ax.set_ylabel("Kumaraswamy b")
    ax.set_title("Kumaraswamy Shape Parameters by Land Cover")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_kuma_by_landcover(master_csv)

In [30]:
def compute_well_totals(all_event_tables, max_ratio=MAX_RATIO,
                           precip_col="precip_total", recharge_col="recharge_total"):
    rows = []
    for well_id, table in all_event_tables.items():
        df = filter_events_by_ratio(table, max_ratio=max_ratio,
                                       precip_col=precip_col, recharge_col=recharge_col)
        if len(df) == 0:
            continue
        total_precip = df[precip_col].sum()
        total_recharge = df[recharge_col].sum() * 1000  # convert to mm
        rows.append({
            "well_id": well_id,
            "n_events": len(df),
            "total_precip_mm": total_precip,
            "total_recharge_mm": total_recharge,
            "recharge_efficiency": total_recharge / total_precip if total_precip > 0 else None,
        })
    return pd.DataFrame(rows)


new_totals_df = compute_well_totals(all_event_tables, max_ratio=MAX_RATIO)

master_csv["well_id"] = master_csv["well_id"].astype(str)
new_totals_df["well_id"] = new_totals_df["well_id"].astype(str)

# Drop the stale/duplicate columns, replace with the corrected, single version
master_csv = master_csv.drop(columns=["n_events_x", "n_events_y", "total_precip_mm",
                                        "total_recharge_mm", "recharge_efficiency"], errors="ignore")
master_csv = master_csv.merge(new_totals_df, on="well_id", how="left")

master_csv.to_csv("master_well_summary.csv", index=False)

print(f"Updated: {len(master_csv)} rows, {len(master_csv.columns)} columns")
print(master_csv.columns.tolist())

Updated: 163 rows, 21 columns
['well_id', 'mtpi', 'elevation_m', 'mean_depth_to_water_m', 'clay', 'sand', 'silt', 'state', 'precip_gini', 'recharge_gini', 'kuma_a', 'kuma_b', 'lat', 'lon', 'nlcd_code', 'landcover', 'landcover_simple', 'n_events', 'total_precip_mm', 'total_recharge_mm', 'recharge_efficiency']


#### Developed land has lower recharge efficiency than forested land.

In [31]:
from scipy import stats

df = master_csv.dropna(subset=["recharge_efficiency", "landcover_simple"])

print(df.groupby("landcover_simple")["recharge_efficiency"].describe())

categories = list(df["landcover_simple"].unique())
data_by_cat = [df[df["landcover_simple"] == c]["recharge_efficiency"].values for c in categories]

fig, ax = plt.subplots(figsize=(9, 6))
ax.boxplot(data_by_cat, tick_labels=categories)
ax.set_ylabel("Recharge Efficiency")
ax.set_title("Recharge Efficiency by Land Cover")
ax.tick_params(axis="x", rotation=30)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()
# Direct comparison: Developed vs Forest specifically
developed = df[df["landcover_simple"] == "Developed"]["recharge_efficiency"]
forest = df[df["landcover_simple"] == "Forest"]["recharge_efficiency"]

if len(developed) >= 3 and len(forest) >= 3:
    t_stat, p_val = stats.ttest_ind(developed, forest)
    print(f"\nDeveloped (n={len(developed)}) mean: {developed.mean():.4f}")
    print(f"Forest (n={len(forest)}) mean: {forest.mean():.4f}")
    print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")
else:
    print(f"\nToo few wells: Developed n={len(developed)}, Forest n={len(forest)}")

                  count      mean       std       min       25%       50%  \
landcover_simple                                                            
Agriculture        22.0  4.244035  1.403764  2.195961  3.021888  4.386414   
Barren              1.0  0.832170       NaN  0.832170  0.832170  0.832170   
Developed          82.0  3.276203  1.594377  0.702448  1.828468  3.425349   
Forest             41.0  3.786863  1.535226  1.106350  2.688274  3.956850   
Grassland           2.0  1.826926  1.199055  0.979066  1.402996  1.826926   
Shrub               1.0  3.637018       NaN  3.637018  3.637018  3.637018   
Wetland            14.0  3.566202  1.293848  1.316056  2.568383  3.456018   

                       75%       max  
landcover_simple                      
Agriculture       5.467674  6.423155  
Barren            0.832170  0.832170  
Developed         4.395278  7.528759  
Forest            4.642887  7.375202  
Grassland         2.250856  2.674786  
Shrub             3.637018  3.637

In [32]:
developed = df[df["landcover_simple"] == "Developed"]["recharge_efficiency"]
forest = df[df["landcover_simple"] == "Forest"]["recharge_efficiency"]

t_stat, p_val = stats.ttest_ind(developed, forest)
print(f"Developed (n={len(developed)}) mean: {developed.mean():.3f}")
print(f"Forest (n={len(forest)}) mean: {forest.mean():.3f}")
print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")
print(f"{'Significant difference' if p_val < 0.05 else 'Not a significant difference'}")

Developed (n=82) mean: 3.276
Forest (n=41) mean: 3.787
t-test: t=-1.695, p=0.0926
Not a significant difference


#### Developed land results in recharge being driven by a few large events. 

In [33]:
def classify_shape(a, b):
    if pd.isna(a) or pd.isna(b):
        return None
    if a > 1 and b > 1:
        return "S-shaped (mid-size storms dominate)"
    elif a > 1 and b <= 1:
        return "Late-rise (few biggest storms dominate)"
    elif a <= 1 and b > 1:
        return "Early-rise (small storms dominate)"
    else:
        return "U-shaped (both extremes matter)"

master_csv["kuma_shape"] = master_csv.apply(lambda row: classify_shape(row["kuma_a"], row["kuma_b"]), axis=1)
print(master_csv["kuma_shape"].value_counts())

kuma_shape
Late-rise (few biggest storms dominate)    68
U-shaped (both extremes matter)            43
S-shaped (mid-size storms dominate)        34
Early-rise (small storms dominate)         18
Name: count, dtype: int64


In [34]:
df = master_csv.dropna(subset=["kuma_shape", "landcover_simple"])

crosstab = pd.crosstab(df["landcover_simple"], df["kuma_shape"])
print(crosstab)

# Row percentages -- what % of each land cover type falls in each shape category
crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100
print("\nAs percentages within each land cover type:")
print(crosstab_pct.round(1))

kuma_shape        Early-rise (small storms dominate)  \
landcover_simple                                       
Agriculture                                        0   
Barren                                             1   
Developed                                          8   
Forest                                             6   
Grassland                                          1   
Shrub                                              0   
Wetland                                            2   

kuma_shape        Late-rise (few biggest storms dominate)  \
landcover_simple                                            
Agriculture                                            13   
Barren                                                  0   
Developed                                              33   
Forest                                                 16   
Grassland                                               1   
Shrub                                                   1   
Wetland

In [35]:
df["is_late_rise"] = df["kuma_shape"] == "Late-rise (few biggest storms dominate)"
df["is_developed"] = df["landcover_simple"] == "Developed"

contingency = pd.crosstab(df["is_developed"], df["is_late_rise"])
print(contingency)

chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-square test: chi2={chi2:.3f}, p={p:.4f}")
print(f"{'Significant association' if p < 0.05 else 'No significant association'}")

pct_developed_late = (df[df["is_developed"]]["is_late_rise"].mean()) * 100
pct_other_late = (df[~df["is_developed"]]["is_late_rise"].mean()) * 100
print(f"\n% of Developed wells that are Late-rise: {pct_developed_late:.1f}%")
print(f"% of other wells that are Late-rise: {pct_other_late:.1f}%")

is_late_rise  False  True 
is_developed              
False            46     35
True             49     33

Chi-square test: chi2=0.051, p=0.8219
No significant association

% of Developed wells that are Late-rise: 40.2%
% of other wells that are Late-rise: 43.2%


In [36]:
fig, ax = plt.subplots(figsize=(10, 6))
crosstab_pct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_ylabel("% of wells in category")
ax.set_xlabel("Land Cover Type")
ax.set_title("Kumaraswamy Shape Category by Land Cover")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=30, ha="right")
fig.tight_layout()
plt.show()

#### Agricultural recharge is dominated by a few large storms. 

In [37]:
df = master_csv.dropna(subset=["kuma_shape", "landcover_simple"])

df["is_late_rise"] = df["kuma_shape"] == "Late-rise (few biggest storms dominate)"
df["is_agriculture"] = df["landcover_simple"] == "Agriculture"

contingency = pd.crosstab(df["is_agriculture"], df["is_late_rise"])
print(contingency)

chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-square test: chi2={chi2:.3f}, p={p:.4f}")
print(f"{'Significant association' if p < 0.05 else 'No significant association'}")

pct_ag_late = (df[df["is_agriculture"]]["is_late_rise"].mean()) * 100
pct_other_late = (df[~df["is_agriculture"]]["is_late_rise"].mean()) * 100
print(f"\n% of Agricultural wells that are Late-rise: {pct_ag_late:.1f}%")
print(f"% of other wells that are Late-rise: {pct_other_late:.1f}%")

n_ag = df["is_agriculture"].sum()
print(f"\n(n={n_ag} Agricultural wells)")


is_late_rise    False  True 
is_agriculture              
False              86     55
True                9     13

Chi-square test: chi2=2.385, p=0.1225
No significant association

% of Agricultural wells that are Late-rise: 59.1%
% of other wells that are Late-rise: 39.0%

(n=22 Agricultural wells)


#### MTPI affects whether a certain storm magnitude dominates.

In [38]:
df = master_csv.dropna(subset=["mtpi", "kuma_shape"])

print(df.groupby("kuma_shape")["mtpi"].describe())

fig, ax = plt.subplots(figsize=(10, 6))
categories = list(df["kuma_shape"].unique())
data_by_cat = [df[df["kuma_shape"] == c]["mtpi"].values for c in categories]
ax.boxplot(data_by_cat, tick_labels=categories)
ax.set_ylabel("mTPI (negative = valley, positive = ridge)")
ax.set_title("Topographic Position by Kumaraswamy Shape Category")
ax.tick_params(axis="x", rotation=20)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

                                         count      mean        std   min  \
kuma_shape                                                                  
Early-rise (small storms dominate)        18.0 -3.000000   7.798944 -29.0   
Late-rise (few biggest storms dominate)   68.0 -9.338235  14.192294 -60.0   
S-shaped (mid-size storms dominate)       34.0 -4.264706   9.991842 -36.0   
U-shaped (both extremes matter)           43.0 -2.000000  12.266874 -54.0   

                                          25%  50%  75%   max  
kuma_shape                                                     
Early-rise (small storms dominate)       -1.0  0.0  0.0   3.0  
Late-rise (few biggest storms dominate) -14.5 -5.0  0.0  19.0  
S-shaped (mid-size storms dominate)      -8.0 -1.0  0.0  14.0  
U-shaped (both extremes matter)          -6.5  0.0  1.0  25.0  


#### Late rise wills have a higher mtpi than when both both extremes matter. 

In [39]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_result = pairwise_tukeyhsd(df["mtpi"], df["kuma_shape"], alpha=0.05)
print(tukey_result)

                                  Multiple Comparison of Means - Tukey HSD, FWER=0.05                                  
                 group1                                  group2                 meandiff p-adj   lower    upper  reject
-----------------------------------------------------------------------------------------------------------------------
     Early-rise (small storms dominate) Late-rise (few biggest storms dominate)  -6.3382 0.2154  -14.819  2.1426  False
     Early-rise (small storms dominate)     S-shaped (mid-size storms dominate)  -1.2647  0.985 -10.5909  8.0615  False
     Early-rise (small storms dominate)         U-shaped (both extremes matter)      1.0 0.9916   -7.982   9.982  False
Late-rise (few biggest storms dominate)     S-shaped (mid-size storms dominate)   5.0735 0.2076  -1.6467 11.7938  False
Late-rise (few biggest storms dominate)         U-shaped (both extremes matter)   7.3382 0.0139   1.1045  13.572   True
    S-shaped (mid-size storms dominate) 

In [40]:
df = master_csv.dropna(subset=["mean_depth_to_water_m", "kuma_shape"])

print(df.groupby("kuma_shape")["mean_depth_to_water_m"].agg(["mean", "median", "std", "count"]))

fig, ax = plt.subplots(figsize=(10, 6))
categories = list(df["kuma_shape"].unique())
data_by_cat = [df[df["kuma_shape"] == c]["mean_depth_to_water_m"].values for c in categories]
ax.boxplot(data_by_cat, tick_labels=categories)
ax.set_ylabel("Mean depth to water table (m)")
ax.set_title("Depth to Water Table by Kumaraswamy Shape Category")
ax.tick_params(axis="x", rotation=20)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

                                              mean    median       std  count
kuma_shape                                                                   
Early-rise (small storms dominate)       10.045168  8.657592  7.420599     18
Late-rise (few biggest storms dominate)   4.117235  2.817391  3.974282     68
S-shaped (mid-size storms dominate)       3.267793  2.158085  3.107975     34
U-shaped (both extremes matter)           7.510911  4.810177  8.331150     43


In [41]:
groups = [df[df["kuma_shape"] == c]["mean_depth_to_water_m"].values for c in categories if len(df[df["kuma_shape"] == c]) >= 3]

if len(groups) >= 2:
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"ANOVA: F={f_stat:.3f}, p={p_val:.4f}")
    print(f"{'Significant difference across categories' if p_val < 0.05 else 'No significant difference across categories'}")
    

ANOVA: F=8.567, p=0.0000
Significant difference across categories


In [42]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_result = pairwise_tukeyhsd(df["mean_depth_to_water_m"], df["kuma_shape"], alpha=0.05)
print(tukey_result)

                                  Multiple Comparison of Means - Tukey HSD, FWER=0.05                                  
                 group1                                  group2                 meandiff p-adj   lower    upper  reject
-----------------------------------------------------------------------------------------------------------------------
     Early-rise (small storms dominate) Late-rise (few biggest storms dominate)  -5.9279 0.0008  -9.8744 -1.9814   True
     Early-rise (small storms dominate)     S-shaped (mid-size storms dominate)  -6.7774 0.0005 -11.1173 -2.4375   True
     Early-rise (small storms dominate)         U-shaped (both extremes matter)  -2.5343 0.3962   -6.714  1.6455  False
Late-rise (few biggest storms dominate)     S-shaped (mid-size storms dominate)  -0.8494 0.8949  -3.9767  2.2778  False
Late-rise (few biggest storms dominate)         U-shaped (both extremes matter)   3.3937 0.0147   0.4928  6.2945   True
    S-shaped (mid-size storms dominate) 

In [43]:
early_rise_wells = master_csv[master_csv["kuma_shape"] == "Early-rise (small storms dominate)"]
print(early_rise_wells[["clay", "sand", "silt"]].mean())
print(master_csv[["clay", "sand", "silt"]].mean())  # compare to overall average

clay    11.978571
sand    58.292857
silt    29.742857
dtype: float64
clay    17.182443
sand    42.900763
silt    39.919847
dtype: float64


In [44]:
print(master_csv[master_csv["kuma_shape"] == "Early-rise (small storms dominate)"][["well_id", "mean_depth_to_water_m"]].sort_values("mean_depth_to_water_m", ascending=False))

             well_id  mean_depth_to_water_m
93   414159070310501              29.196771
35   401834074515501              20.549142
53   404518077575501              20.472123
25   400120074265401              15.436183
55   404639074230001              13.263529
94   414159079213601              12.392171
92   414129070361401              10.035569
98   414632070014901               9.388223
15   394440074593101               8.968305
91   414101070011001               8.346879
20   395150074284201               6.801210
19   395122074301702               6.336895
10   394106074362501               4.004685
103  415354069585201               3.555375
105  420206070045901               3.492925
47   403455074514801               3.080590
6    392920074570001               2.762751
26   400148074352101               2.729705


In [45]:
early_rise_depths = master_csv[master_csv["kuma_shape"] == "Early-rise (small storms dominate)"]["mean_depth_to_water_m"]

print(f"Mean: {early_rise_depths.mean():.2f}")
print(f"Median: {early_rise_depths.median():.2f}")

# What happens if we drop just the 3 most extreme wells?
without_top3 = early_rise_depths.sort_values(ascending=False).iloc[3:]
print(f"\nMean without top 3 deepest: {without_top3.mean():.2f}")
print(f"Median without top 3 deepest: {without_top3.median():.2f}")

Mean: 10.05
Median: 8.66

Mean without top 3 deepest: 7.37
Median without top 3 deepest: 6.80


#### Overall seems like the early rise wells are sandier

In [46]:
early_rise_wells = master_csv[master_csv["kuma_shape"] == "Early-rise (small storms dominate)"]
print("Early-rise wells' soil texture:")
print(early_rise_wells[["clay", "sand", "silt"]].mean())

print("\nOverall average soil texture:")
print(master_csv[["clay", "sand", "silt"]].mean())

Early-rise wells' soil texture:
clay    11.978571
sand    58.292857
silt    29.742857
dtype: float64

Overall average soil texture:
clay    17.182443
sand    42.900763
silt    39.919847
dtype: float64


#### A high preccipitation recharge gini means less effective recharge. (Are all my ratios higher because rpsy is not recharge but actually it is total change in water table

In [47]:
from scipy import stats

df = master_csv.dropna(subset=["precip_gini", "recharge_efficiency"])

r, p = stats.pearsonr(df["precip_gini"], df["recharge_efficiency"])
print(f"Correlation: r={r:.3f}, p={p:.4f}, n={len(df)}")
print(f"{'Significant' if p < 0.05 else 'Not significant'}")

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(df["precip_gini"], df["recharge_efficiency"], alpha=0.6, s=40,
           color="tab:orange", edgecolor="black", linewidth=0.3)

coeffs = np.polyfit(df["precip_gini"], df["recharge_efficiency"], 1)
x_line = np.linspace(df["precip_gini"].min(), df["precip_gini"].max(), 50)


ax.set_xlabel("Precipitation Gini (higher = rain more concentrated in few storms)")
ax.set_ylabel("Recharge Efficiency")
ax.set_title(f"Precipitation Gini vs Recharge Efficiency\nr = {r:.3f}, p = {p:.4f}, n = {len(df)}")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Correlation: r=-0.008, p=0.9195, n=163
Not significant


In [ ]:
df = master_csv.dropna(subset=["precip_gini", "recharge_efficiency"]).copy()

df["gini_group"] = pd.qcut(df["precip_gini"], 3, labels=["Low concentration", "Medium concentration", "High concentration"])

print(df.groupby("gini_group")["recharge_efficiency"].agg(["mean", "median", "std", "count"]))

fig, ax = plt.subplots(figsize=(9, 6))
groups = ["Low concentration", "Medium concentration", "High concentration"]
data_by_group = [df[df["gini_group"] == g]["recharge_efficiency"].values for g in groups]
ax.boxplot(data_by_group, tick_labels=groups)
ax.set_ylabel("Recharge Efficiency")
ax.set_title("Recharge Efficiency by Precipitation Concentration Group")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

# Direct comparison: lowest third vs highest third
low = df[df["gini_group"] == "Low concentration"]["recharge_efficiency"]
high = df[df["gini_group"] == "High concentration"]["recharge_efficiency"]

t_stat, p_val = stats.ttest_ind(low, high)
print(f"\nLow concentration (n={len(low)}) mean: {low.mean():.3f}")
print(f"High concentration (n={len(high)}) mean: {high.mean():.3f}")
print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")